# 01. Деректерді дайындау

Бұл ноутбук оқыту үшін деректер жинайды.

### Label=0 (сенімді жаңалықтар):
- **RSS фидтер** — lenta.ru, tass.ru, tengrinews.kz, BBC Russian және т.б.
- **Синтетикалық сенімді мәтіндер** — нақты жаңалықтар стилінде

### Label=1 (дезинформация):
- **factcheck.kz** — WordPress API (жалған жаңалықтар)
- **Синтетика** — рандомды құрамдас бөліктерден жиналған жалған мәтіндер (RU + KZ)

### MVP нұсқасы
- RSS + синтетика = ~3000-5000 жазба
- Уақыты: ~10-15 минут
- **TODO (қорғауға):** tengrinews.kz sitemap скрапинг (~2000), informburo.kz (~1000)

In [ ]:
# 1. Тәуелділіктерді орнату
!pip install -q feedparser beautifulsoup4 lxml requests pandas tqdm datasets

In [ ]:
# 2. Google Drive қосу
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/disinformation_project'
os.makedirs(f'{SAVE_DIR}/data', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/models', exist_ok=True)
print(f'Деректер сақталатын жер: {SAVE_DIR}')

In [ ]:
# 3. Конфигурация және көмекші функциялар
import re
import time
import random
import requests
import pandas as pd
import feedparser
from bs4 import BeautifulSoup
from tqdm import tqdm
from xml.etree import ElementTree as ET

random.seed(42)

# Баптаулар
MAX_TEXT = 1000        # мақала үшін макс символдар
MIN_WORDS = 20         # мин сөздер саны
REQUEST_DELAY = 0.5    # сұраныстар арасындағы кідіріс (сек)
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; DisinfoResearchBot/1.0)'}
TIMEOUT = 30

data = []  # барлық жазбаларды осында жинаймыз

def clean_html(html):
    """HTML-тегтерді алып тастайды, таза мәтін қайтарады."""
    if not html:
        return ''
    soup = BeautifulSoup(html, 'lxml')
    return soup.get_text(separator=' ', strip=True)

def clean_text(text):
    """URL-дерді, артық бос орындарды алып тастайды."""
    if not text:
        return ''
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:MAX_TEXT]

def is_valid(text):
    return len(text.split()) >= MIN_WORDS

def fetch(url, retries=3):
    """GET-сұраныс, қайталау мүмкіндігімен."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if resp.status_code == 200:
                return resp
            if resp.status_code == 429:
                time.sleep(5 * (attempt + 1))
                continue
        except Exception as e:
            if attempt == retries - 1:
                print(f'  Қате {url}: {e}')
        time.sleep(2 * (attempt + 1))
    return None

def save_checkpoint(label='checkpoint'):
    """Аралық нәтижені сақтайды."""
    df_tmp = pd.DataFrame(data)
    df_tmp.to_csv(f'{SAVE_DIR}/data/{label}.csv', index=False)
    print(f'  Чекпоинт: {len(data)} жазба сақталды')

print('Көмекші функциялар жүктелді. Деректер жинауды бастаймыз...')

## Label=0: Сенімді жаңалықтар

In [ ]:
# ============================================================
# TODO (қорғауға): tengrinews.kz Sitemap скрапинг (~2000 мақала)
# Толық нұсқада осы ұяшықты іске қосыңыз
# Уақыты: ~15-20 минут
# ============================================================

# print('=== tengrinews.kz (Sitemap) ===')
# TENGRI_TARGET = 2000
# ... sitemap скрапинг коды ...

print('⏭ tengrinews.kz sitemap — MVP-де өткізілді (қорғауда іске қосылады)')

In [ ]:
# ============================================================
# TODO (қорғауға): informburo.kz Sitemap скрапинг (~1000 мақала)
# Толық нұсқада осы ұяшықты іске қосыңыз
# Уақыты: ~10 минут
# ============================================================

# print('=== informburo.kz (Sitemap) ===')
# INFORMBURO_TARGET = 1000
# ... sitemap скрапинг коды ...

print('⏭ informburo.kz sitemap — MVP-де өткізілді (қорғауда іске қосылады)')

In [ ]:
# 4. Label=0: RSS фидтер (нақты жаңалықтар)
print('=== RSS фидтер ===')

RSS_FEEDS = [
    # Lenta.ru
    ('lenta_news',      'https://lenta.ru/rss/news'),
    ('lenta_articles',  'https://lenta.ru/rss/articles'),
    ('lenta_russia',    'https://lenta.ru/rss/news/russia'),
    ('lenta_world',     'https://lenta.ru/rss/news/world'),
    ('lenta_economics', 'https://lenta.ru/rss/news/economics'),
    ('lenta_science',   'https://lenta.ru/rss/news/science'),
    # Kommersant
    ('kommersant_news',      'https://www.kommersant.ru/RSS/news.xml'),
    ('kommersant_politics',  'https://www.kommersant.ru/RSS/section-politics.xml'),
    ('kommersant_economics', 'https://www.kommersant.ru/RSS/section-economics.xml'),
    ('kommersant_society',   'https://www.kommersant.ru/RSS/section-society.xml'),
    # Vedomosti
    ('vedomosti_news',     'https://www.vedomosti.ru/rss/news.xml'),
    ('vedomosti_articles', 'https://vedomosti.ru/rss/articles.xml'),
    # Басқалары
    ('ria',       'https://ria.ru/export/rss2/archive/index.xml'),
    ('tass',      'https://tass.ru/rss/v2.xml'),
    ('interfax',  'https://www.interfax.ru/rss.asp'),
    ('bbc_ru',    'https://feeds.bbci.co.uk/russian/rss.xml'),
    ('kursiv_kz', 'https://kz.kursiv.media/feed/'),
    ('tengri_rss','https://tengrinews.kz/news.rss'),
]

total_rss = 0
for name, url in RSS_FEEDS:
    try:
        feed = feedparser.parse(url)
        count = 0
        for entry in feed.entries:
            title = entry.get('title', '')
            summary = ''
            if hasattr(entry, 'content') and entry.content:
                summary = entry.content[0].get('value', '')
            elif hasattr(entry, 'summary'):
                summary = entry.get('summary', '')
            text = clean_html(f'{title}. {summary}' if title else summary)
            text = clean_text(text)
            if is_valid(text):
                data.append({'text': text, 'label': 0, 'source': f'rss_{name}'})
                count += 1
        total_rss += count
        print(f'  {name}: {count} мақала')
    except Exception as e:
        print(f'  {name}: ҚАТЕ — {e}')

print(f'\nRSS-тен барлығы: {total_rss}')
save_checkpoint('after_rss')

In [ ]:
# 5. Label=0: nur.kz фид
print('=== nur.kz ===')

try:
    resp = fetch('https://www.nur.kz/feed/')
    if resp:
        count = 0
        lines = resp.text.strip().split('\n')
        for line in lines:
            parts = line.split(',')
            if len(parts) >= 2:
                headline = parts[1].strip().strip('"')
                text = clean_text(headline)
                if len(text.split()) >= 5:
                    data.append({'text': text, 'label': 0, 'source': 'nur_kz'})
                    count += 1
        print(f'  nur.kz: {count} жазба')
    else:
        print('  nur.kz: жүктелмеді')
except Exception as e:
    print(f'  nur.kz: ҚАТЕ — {e}')

# Label=0 аралық статистика
label0_count = sum(1 for d in data if d['label'] == 0)
print(f'\n=== Label=0 барлығы: {label0_count} жазба ===')
save_checkpoint('after_all_label0')

## Label=1: Дезинформация (нақты + синтетика)

In [ ]:
# 6. Label=1: factcheck.kz (WordPress REST API)
print('=== factcheck.kz (WordPress API) ===')

factcheck_count = 0
page = 1

while True:
    url = f'https://factcheck.kz/wp-json/wp/v2/posts?per_page=100&page={page}'
    resp = fetch(url)
    if not resp or resp.status_code != 200:
        if page == 1:
            print('  API қол жетімді емес, мұрағатты скрапинг жасаймыз...')
            for archive_page in range(1, 50):
                archive_url = f'https://factcheck.kz/page/{archive_page}/'
                resp2 = fetch(archive_url)
                if not resp2 or resp2.status_code != 200:
                    break
                soup = BeautifulSoup(resp2.content, 'lxml')
                articles = soup.find_all('article') or soup.find_all('div', class_='post')
                if not articles:
                    break
                for art in articles:
                    title = art.find(['h2', 'h3'])
                    title_text = title.get_text(strip=True) if title else ''
                    excerpt = art.find('div', class_='entry-content') or art.find('p')
                    excerpt_text = excerpt.get_text(strip=True) if excerpt else ''
                    text = clean_text(f'{title_text}. {excerpt_text}')
                    if is_valid(text):
                        data.append({'text': text, 'label': 1, 'source': 'factcheck_kz'})
                        factcheck_count += 1
                time.sleep(REQUEST_DELAY)
        break

    try:
        posts = resp.json()
    except:
        break

    if not posts:
        break

    for post in posts:
        title = clean_html(post.get('title', {}).get('rendered', ''))
        body = clean_html(post.get('content', {}).get('rendered', ''))
        text = clean_text(f'{title}. {body}' if title else body)
        if is_valid(text):
            data.append({'text': text, 'label': 1, 'source': 'factcheck_kz'})
            factcheck_count += 1

    page += 1
    time.sleep(REQUEST_DELAY)

print(f'  factcheck.kz: {factcheck_count} жазба')

In [ ]:
# 7. Label=1: stopfake.org (WordPress REST API)
print('=== stopfake.org (WordPress API) ===')

stopfake_count = 0
page = 1

while True:
    url = f'https://www.stopfake.org/wp-json/wp/v2/posts?per_page=100&page={page}'
    resp = fetch(url)
    if not resp or resp.status_code != 200:
        if page == 1:
            print('  API қол жетімді емес, RSS қолданамыз...')
            try:
                feed = feedparser.parse('http://www.stopfake.org/ru/feed/')
                for entry in feed.entries:
                    title = entry.get('title', '')
                    summary = ''
                    if hasattr(entry, 'content') and entry.content:
                        summary = entry.content[0].get('value', '')
                    elif hasattr(entry, 'summary'):
                        summary = entry.get('summary', '')
                    text = clean_html(f'{title}. {summary}')
                    text = clean_text(text)
                    if is_valid(text):
                        data.append({'text': text, 'label': 1, 'source': 'stopfake'})
                        stopfake_count += 1
                print(f'  RSS-тен: {stopfake_count} жазба')
            except Exception as e:
                print(f'  RSS де жұмыс істемейді: {e}')
        break

    try:
        posts = resp.json()
    except:
        break

    if not posts:
        break

    for post in posts:
        title = clean_html(post.get('title', {}).get('rendered', ''))
        body = clean_html(post.get('content', {}).get('rendered', ''))
        text = clean_text(f'{title}. {body}' if title else body)
        if is_valid(text):
            data.append({'text': text, 'label': 1, 'source': 'stopfake'})
            stopfake_count += 1

    page += 1
    time.sleep(REQUEST_DELAY)
    if stopfake_count > 0 and stopfake_count % 500 == 0:
        save_checkpoint('stopfake_partial')

print(f'  stopfake.org: {stopfake_count} жазба')
save_checkpoint('after_factcheck')

In [ ]:
# 8. Label=1: Жетілдірілген синтетика (рандомды құрастыру)
print('=== Синтетика (рандомды құрастыру) ===')

# Рандомды құрастыру үшін құрамдас бөліктер
prefixes_ru = [
    'ШОК!', 'СРОЧНО!', 'ВНИМАНИЕ!', 'МОЛНИЯ!', 'СЕНСАЦИЯ!',
    'ВАЖНО!', 'ЭКСТРЕННО!', '', '', '', '',
]

intros_ru = [
    'Стало известно, что', 'По данным инсайдеров,', 'Как выяснилось,',
    'Источники сообщают:', 'Эксперты выяснили, что', 'Инсайдеры утверждают:',
    'По непроверенным данным,', 'Из достоверных источников стало известно:',
    'Анонимный сотрудник раскрыл:', 'Утечка документов показала, что',
    'Независимые журналисты узнали:', 'Бывший чиновник рассказал:',
    '', '', '',
]

topics_ru = [
    'цены на продукты вырастут в 3 раза до конца месяца',
    'новая вакцина вызывает необратимые побочные эффекты',
    'курс тенге обвалится до 1000 за доллар',
    'Акорда готовит тайный указ о повышении пенсионного возраста',
    'КНБ начал массовую слежку через приложение eGov',
    'Байконур отравляет грунтовые воды на 500 км вокруг',
    'Аральское море высохло из-за секретных военных экспериментов',
    'уран из Казахстана тайно вывозится в третьи страны',
    'воду в Астане специально заражают химикатами',
    'школы переведут на платное обучение с сентября',
    'пенсии заморозят на 5 лет без индексации',
    'чипы в новых удостоверениях следят за перемещениями граждан',
    '5G вышки в Алматы вызывают головные боли и бессонницу',
    'золотой запас Нацбанка тайно вывезен за рубеж',
    'правительство скрывает реальный уровень безработицы',
    'новый налог отберёт 30% зарплаты у каждого казахстанца',
    'в больницах закончились все лекарства и никто не говорит',
    'продукты в магазинах содержат опасные добавки',
    'банки тайно списывают деньги со счетов казахстанцев',
    'президент тяжело болен но это скрывают от народа',
    'элиты готовятся эвакуироваться из страны',
    'армию готовят к подавлению мирных протестов',
    'в Казахстане строят секретные военные базы иностранных государств',
    'детей в школах заставляют принимать экспериментальные препараты',
    'природные ресурсы Казахстана распроданы иностранцам за бесценок',
    'новый закон запретит критику власти в интернете',
    'тарифы на ЖКХ вырастут в 5 раз без предупреждения',
    'ГМО продукты из Китая заполонили казахстанские магазины',
    'врачи получают бонусы за неправильные диагнозы',
    'результаты выборов были полностью сфальсифицированы',
    'олигархи контролируют все СМИ и манипулируют мнением народа',
    'санкции приведут к полному коллапсу экономики за месяц',
]

appeals_ru = [
    'Власти молчат!', 'Это скрывают от нас.', 'Официальные СМИ замалчивают.',
    'Народ должен знать правду!', 'Почему об этом молчат?',
    'Марионеточные СМИ не расскажут.', 'Проснитесь, люди!',
    'Задумайтесь, кому это выгодно.', 'Это не случайность — это план.',
    'Доколе это будет продолжаться?', 'Они думают мы не заметим.',
    '', '', '',
]

ctas_ru = [
    'Перешлите всем!', 'Максимальный репост!', 'Расскажите близким!',
    'Делитесь, пока не удалили!', 'Сохраните себе и отправьте друзьям!',
    'Читайте, пока не заблокировали!', 'Репост — наше оружие!',
    '', '', '', '',
]

# Қазақша құрамдас бөліктер
prefixes_kz = ['ШҰҒЫЛ!', 'НАЗАР АУДАРЫҢЫЗ!', 'СЕНСАЦИЯ!', '', '', '']

intros_kz = [
    'Белгілі болды:', 'Инсайдерлер хабарлайды:', 'Деректер бойынша,',
    'Жасырын ақпарат:', 'Тексеру нәтижесінде анықталды:',
    '', '',
]

topics_kz = [
    'теңге құны ай соңына дейін 5 есе құлдырайды',
    'Ақорда зейнетақы жасын жасырын көтеруге дайындалуда',
    'ҰҚК eGov арқылы азаматтарды бақылау жүйесін іске қосты',
    'Байқоңыр қоршаған ортаны 500 км радиуста ластайды',
    'Арал теңізі құпия әскери тәжірибелер салдарынан құрғады',
    'мектептерде тегін білім беру тоқтатылады',
    'су құбырындағы су арнайы улы заттармен ластануда',
    'сайлау нәтижелері толығымен бұрмаланған',
    'банктер қазақстандықтардың шоттарынан ақшаны жасырын алып жатыр',
    'жаңа заң интернетте билікті сынауға тыйым салады',
    'ауруханаларда дәрі-дәрмек таусылған, ешкім айтпайды',
    'азық-түлік бағасы 3 есеге өседі',
    'балаларды мектептерде тәжірибелік препараттар қабылдауға мәжбүрлейді',
    'Қазақстанның табиғи ресурстары шетелдіктерге тиынға сатылған',
    'ЖКХ тарифтері ескертусіз 5 есеге өседі',
]

appeals_kz = [
    'Билік үндемей отыр!', 'Бұл бізден жасырылуда.', 'Халық шындықты білуі керек!',
    'Неге бұл туралы айтылмайды?', 'Ояныңыздар, адамдар!',
    '', '', '',
]

ctas_kz = [
    'Барлығына жіберіңіз!', 'Репост жасаңыз!', 'Жақындарыңызға айтыңыз!',
    'Оқыңыз, жойылмай тұрғанда!',
    '', '', '',
]

def generate_fake(lang='ru'):
    """Кездейсоқ құрамдас бөліктерден бір жалған мәтін генерациялайды."""
    if lang == 'ru':
        parts = [
            random.choice(prefixes_ru),
            random.choice(intros_ru),
            random.choice(topics_ru),
            random.choice(appeals_ru),
            random.choice(ctas_ru),
        ]
    else:
        parts = [
            random.choice(prefixes_kz),
            random.choice(intros_kz),
            random.choice(topics_kz),
            random.choice(appeals_kz),
            random.choice(ctas_kz),
        ]
    parts = [p for p in parts if p]
    text = ' '.join(parts)
    if random.random() < 0.3:
        text += '!' * random.randint(1, 3)
    if random.random() < 0.2:
        text = text.upper() if random.random() < 0.3 else text
    return text

# Баланс үшін қанша синтетика керек екенін анықтаймыз
real_label1 = sum(1 for d in data if d['label'] == 1)
label0_total = sum(1 for d in data if d['label'] == 0)
target_label1 = int(label0_total / 1.3)  # баланс 1.3:1
synthetic_needed = max(500, target_label1 - real_label1)  # кемінде 500

print(f'Label=0: {label0_total}, Нақты label=1: {real_label1}')
print(f'Баланс үшін синтетика қажет: {synthetic_needed}')

# RU (70%) және KZ (30%) генерациялаймыз
synthetic_ru = int(synthetic_needed * 0.7)
synthetic_kz = synthetic_needed - synthetic_ru

generated = set()
count = 0
attempts = 0
while count < synthetic_ru and attempts < synthetic_ru * 3:
    text = generate_fake('ru')
    if text not in generated and is_valid(text):
        data.append({'text': text, 'label': 1, 'source': 'synthetic_ru'})
        generated.add(text)
        count += 1
    attempts += 1
print(f'  Синтетика RU: {count}')

count = 0
attempts = 0
while count < synthetic_kz and attempts < synthetic_kz * 3:
    text = generate_fake('kz')
    if text not in generated and is_valid(text):
        data.append({'text': text, 'label': 1, 'source': 'synthetic_kz'})
        generated.add(text)
        count += 1
    attempts += 1
print(f'  Синтетика KZ: {count}')

print(f'\nLabel=1 барлығы: {sum(1 for d in data if d["label"]==1)}')
save_checkpoint('after_synthetic')

In [ ]:
# 9. Тазалау, дедупликация, статистика
print('=== Тазалау және дедупликация ===')

df = pd.DataFrame(data)
print(f'Тазалауға дейін: {len(df)}')

df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.split().str.len() >= MIN_WORDS]
df = df.drop_duplicates(subset='text')
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Тазалаудан кейін: {len(df)}')
print(f'\nКласстар бойынша:')
print(df['label'].value_counts())
print(f'\nДеректер көздері бойынша:')
print(df['source'].value_counts())

In [ ]:
# 10. Баланстау және бөлу (train/val/test)
from sklearn.model_selection import train_test_split

# Баланс: label=0-ді label=1-нің 1.3 еселігіне дейін шектейміз
n1 = len(df[df['label'] == 1])
n0 = len(df[df['label'] == 0])
max_n0 = int(n1 * 1.3)

if n0 > max_n0:
    df0 = df[df['label'] == 0]
    # КЗ көздеріне басымдық береміз
    kz_sources = df0[df0['source'].isin(['tengrinews', 'informburo', 'nur_kz', 'rss_tengri_rss', 'rss_kursiv_kz'])]
    other_sources = df0[~df0['source'].isin(['tengrinews', 'informburo', 'nur_kz', 'rss_tengri_rss', 'rss_kursiv_kz'])]

    if len(kz_sources) >= max_n0:
        df0_balanced = kz_sources.sample(max_n0, random_state=42)
    else:
        remaining = max_n0 - len(kz_sources)
        other_sample = other_sources.sample(min(remaining, len(other_sources)), random_state=42)
        df0_balanced = pd.concat([kz_sources, other_sample])

    df = pd.concat([df0_balanced, df[df['label'] == 1]]).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f'Баланстаудан кейін: {len(df)}')
else:
    print(f'Баланстау қажет емес: {n0} label=0, {n1} label=1')

print(df['label'].value_counts())

# Бөлу 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f'\nTrain: {len(train_df)} ({train_df["label"].mean():.2%} дезинформация)')
print(f'Val:   {len(val_df)} ({val_df["label"].mean():.2%} дезинформация)')
print(f'Test:  {len(test_df)} ({test_df["label"].mean():.2%} дезинформация)')

In [ ]:
# 11. Сақтау
from datasets import Dataset, DatasetDict

# source бағанын алып тастаймыз (модель оны көрмеуі керек)
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True)),
})

save_path = f'{SAVE_DIR}/data/disinfo_dataset'
dataset.save_to_disk(save_path)
print(f'DatasetDict сақталды: {save_path}')
print(dataset)

# Талдау үшін CSV (source бағанымен)
df.to_csv(f'{SAVE_DIR}/data/full_dataset.csv', index=False)
print(f'CSV сақталды: {SAVE_DIR}/data/full_dataset.csv')
print(f'\nДайын! 02_fine_tuning.ipynb ноутбукіне көшіңіз')

In [ ]:
# 12. Визуалды тексеру
import matplotlib.pyplot as plt

print('=== 5 СЕНІМДІ мысал (label=0) ===')
for _, row in df[df['label']==0].sample(5, random_state=42).iterrows():
    print(f'  [{row["source"]}] {row["text"][:120]}...')
    print()

print('=== 5 ДЕЗИНФОРМАЦИЯ мысал (label=1) ===')
for _, row in df[df['label']==1].sample(5, random_state=42).iterrows():
    print(f'  [{row["source"]}] {row["text"][:120]}...')
    print()

# Графиктер
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Класстар таралуы
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Класстар таралуы')
axes[0].set_xticklabels(['Сенімді (0)', 'Дезинформация (1)'], rotation=0)

# 2. Деректер көздері бойынша
df['source'].value_counts().head(15).plot(kind='barh', ax=axes[1])
axes[1].set_title('Топ-15 деректер көздері')

# 3. Мәтін ұзындығы
df['text_len'] = df['text'].str.split().str.len()
df[df['label']==0]['text_len'].hist(ax=axes[2], bins=50, alpha=0.5, label='Сенімді')
df[df['label']==1]['text_len'].hist(ax=axes[2], bins=50, alpha=0.5, label='Дезинформация')
axes[2].set_title('Мәтін ұзындығы (сөз)')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/data/dataset_stats.png', dpi=100)
plt.show()
print('График сақталды!')